# Milestone 1 lab, notebook 2 of 8: Airflow runs it

## Where we are

| Notebook | What you do |
|---|---|
| 1. The question and five green checks | Meet Lotus Retails and the Monday question. Run the given pipeline for one night. Read its five checks |
| **2. Airflow runs it** (you are here) | Let Airflow run the same night. Read the DAG: its tasks and its arrows |
| 3. The number by hand | Add up one store's receipts yourself. Find out why the two numbers differ |
| 4. Who is hurt | For each table: who uses it, and what happens to them when it is wrong |
| 5. The check card | Meet the page you fill in before any check. Say what one row means |
| 6. Read the table first | Count the rows, find the smallest and largest values, count the empty cells |
| 7. Six questions for any table | Sort your worries under six questions. See which ones the five checks answer |
| 8. The production version | The same logic as a tested program |

**You need:** Airflow running. From the project folder: `docker compose up -d --build`. The first start takes a few minutes. The [README](../README.md#set-up) has every step.

**About ten minutes.**

In [1]:
import pandas as pd
from lab_helpers import MILESTONE_DIR, check_setup, run_pipeline, sql

import airflow_tools  # lab_helpers put this milestone's folder on the path, so this is found

check_setup(airflow=True)
airflow_tools.wait_until_ready()

The warehouse is running: Postgres on port 5450.


Airflow is running: http://localhost:8080


## Step 1: Who presses the button every night?

In notebook 1 you ran the pipeline yourself. A real pipeline runs at 02:00, when you are asleep. Something must start each step, in the right order, and stop when a step fails.

That is the job of **Apache Airflow**. You tell it the steps and the order. It does the rest.

Airflow reads that from one Python file. Here are the steps it found in ours.

In [2]:
print(airflow_tools.tasks())

['check', 'clean', 'load', 'publish', 'stores_send_files']


Each of those is a **task**: one step of the pipeline that Airflow starts.

Airflow lists them by name. The order they run in is written on one line of the file.

In [3]:
dag_file = (MILESTONE_DIR / "dags" / "lotus_nightly_dag.py").read_text()
print([line.strip() for line in dag_file.splitlines() if line.strip().startswith("stores_send_files >>")][0])

stores_send_files >> load >> clean >> check >> publish


Read `>>` as "then". The whole line is the **DAG**: the list of the pipeline's tasks and the order they run in.

DAG is short for *directed acyclic graph*. You do not need the long name. It means: boxes, arrows that point one way, and no arrow that leads back to an earlier box.

## Step 2: A task is one command

Look at what the `load` task does.

In [4]:
for line in dag_file.splitlines():
    if line.startswith(("PYTHON =", "PIPELINE =")) or 'task_id="load"' in line:
        print(line.strip())

PYTHON = "/opt/lotus-venv/bin/python"  # the pipeline's own virtual environment, inside the image
PIPELINE = f"cd {MILESTONE_DIR} && {PYTHON} -m lotus_pipeline"
load = BashOperator(task_id="load", bash_command=f"{PIPELINE} load --day {BUSINESS_DAY}")


It starts `python -m lotus_pipeline load --day ...`. That is the same command you can type in a terminal.

**Airflow holds no business logic.** It does not know what a sale is. It knows when to start a command, and what to do when the command fails.

## Step 3: Let Airflow run the night

Ask Airflow to run Monday 24 August. This takes about twenty seconds.

In [5]:
states = airflow_tools.run_night("2026-08-24")
pd.DataFrame(states)

,task,state
0,stores_send_files,success
1,load,success
2,clean,success
3,check,success
4,publish,success


Five tasks, all `success`.

**Now look at it yourself.** Open http://localhost:8080 , click **Dags**, then `lotus_nightly_m1`. You see the five boxes and the run you just made, all green. Click the `publish` box, then **Logs**: the bonus report is there.

## Step 4: The same numbers

You ran this night in notebook 1. Airflow has now run it again. Did anything change?

In [6]:
sql("""
    SELECT store_id, net_sales_cents
    FROM output.daily_store_sales
    WHERE sales_date = '2026-08-24'
    ORDER BY store_id
""")

,store_id,net_sales_cents
0,S01,79524
1,S02,90128
2,S03,79618
3,S04,57977
4,S05,47418
5,S06,59519


In [7]:
sql("SELECT count(*) AS files_loaded FROM raw.loaded_files")

,files_loaded
0,622


`S03` is still 79618 cents: $796.18. And 622 files are loaded, not 1,244. The load step skips a file it has already loaded.

Same commands, same numbers. Airflow only pressed the button.

## Step 5: The arrow is a dependency

An arrow between two tasks is a **dependency**. The second task waits for the first. If the first fails, the second does not run.

Look at this arrow: `check >> publish`. It is the only safety this pipeline has.

See it work. Put one sales line into the clean table twice, by hand, and run `check` and `publish` again.

In [8]:
from lotus_pipeline import pipeline
from lotus_pipeline.warehouse import connect

with connect() as wh:
    wh.execute(
        "INSERT INTO clean.sales_lines SELECT * FROM clean.sales_lines WHERE line_id = ?",
        ("S03-0824-000001",),
    )

try:
    run_pipeline("2026-08-24", ("check", "publish"))
except pipeline.ChecksFailed as failed:
    print(failed)
    print("\nThe check failed, so publish did not run.")

check:   the five standard checks
  1. The table is not empty           clean.sales_lines    PASS
  2. The key is not null              clean.sales_lines    PASS
  3. The key is unique                clean.sales_lines    FAIL (1 bad rows)
  4. The status is an allowed value   clean.sales_lines    PASS
  5. The total is positive            clean.sales_lines    PASS
  4 of 5 checks passed.

The check failed, so publish did not run.


Check 3 found the line you added. The program stopped before `publish`.

Airflow does the same with its arrow. It marks `check` as **failed** and `publish` as **upstream_failed**: a task before it failed, so it was not started. You will see those two states for real in milestone 4.

Now put the table back. The `clean` step builds it again from the raw rows.

In [9]:
run_pipeline("2026-08-24", ("clean", "check"))

clean:   7 clean tables rebuilt. sales_lines has 20,482 rows.
check:   the five standard checks
  1. The table is not empty           clean.sales_lines    PASS
  2. The key is not null              clean.sales_lines    PASS
  3. The key is unique                clean.sales_lines    PASS
  4. The status is an allowed value   clean.sales_lines    PASS
  5. The total is positive            clean.sales_lines    PASS
  5 of 5 checks passed.


So the pipeline is only as safe as its checks. `publish` waits for `check`, and `check` can only stop what it can see.

## What you now know

- A **task** is one step that Airflow starts. A **DAG** is the tasks and their order.
- A **dependency** is an arrow. If the first task fails, the second does not run.
- Airflow holds no business logic. Each task is a command you can type.
- Running a night twice changes nothing.

**Say it in an interview:** "Our nightly DAG had five tasks in a line: the stores send files, load, clean, check, publish. Publish depends on check, so a failed check stops the publish. That arrow was the pipeline's only safety, so everything depended on what the checks could see."

**The theory:** [DAGs, Tasks & Operators](https://www.d8loop.com/modules/knowledge-corner/airflow/dags-tasks-and-operators-the-core-model) and [Writing Your First DAG & Reading the UI](https://www.d8loop.com/modules/knowledge-corner/airflow/writing-your-first-dag-and-reading-the-ui). The full Airflow story is in [Daily Orders](https://github.com/vijaysngrajput/daily-orders-api-to-warehouse), milestone 5.

Next: [notebook 3](03_the_number_by_hand.ipynb). You test the number.